# HIP-4 : passer un ordre sur un marche de prediction Hyperliquid

[Hyperliquid Python SDK](https://github.com/hyperliquid-dex/hyperliquid-python-sdk)
[HIP-4 docs](https://hyperliquid.gitbook.io/hyperliquid-docs/hyperliquid-improvement-proposals-hips/hip-4-outcome-markets)

## 1. Setup

In [ ]:
# pip install hyperliquid-python-sdk eth-account python-dotenv requests

In [ ]:
import os
import math
import requests
from dotenv import load_dotenv
import eth_account

from hyperliquid.info import Info
from hyperliquid.exchange import Exchange
from hyperliquid.utils import constants

load_dotenv()

BASE_URL = constants.MAINNET_API_URL    # https://api.hyperliquid.xyz
# BASE_URL = constants.TESTNET_API_URL  # https://api.hyperliquid-testnet.xyz

PRIVATE_KEY = os.environ["HL_SECRET_KEY"]          # cle privee (confidentielle !)
ACCOUNT_ADDRESS = os.environ["HL_ACCOUNT_ADDRESS"]  # adresse publique du compte

account = eth_account.Account.from_key(PRIVATE_KEY)
info = Info(BASE_URL, skip_ws=True)
hyperliquid = Exchange(account, BASE_URL, account_address=ACCOUNT_ADDRESS)

## 2. Enregistrer les outcomes HIP-4

Le SDK ne charge pas les marches HIP-4 a l'initialisation. On enregistre donc
chaque cote (YES / NO) de chaque outcome dans les tables de correspondance du SDK
avant de pouvoir router un ordre.

Encodage : le coin est la chaine `#N` ou `N = 10 * outcome + side`
(side 0 = YES, side 1 = NO). L'asset id vaut `100_000_000 + N`.

In [ ]:
def register_hip4_outcomes(meta, *clients):
    for o in meta["outcomes"]:
        for side in (0, 1):
            coin = f"#{10 * o['outcome'] + side}"
            asset_id = 100_000_000 + 10 * o["outcome"] + side
            for client in clients:
                client.coin_to_asset[coin] = asset_id
                client.name_to_coin[coin] = coin

meta = requests.post(f"{BASE_URL}/info", json={"type": "outcomeMeta"}).json()
register_hip4_outcomes(meta, info, hyperliquid.info)

## 3. Lister les marches

`outcomeMeta` renvoie deux listes : `questions` (les evenements) et `outcomes`
(chaque issue traçable, avec son numero dans le champ `outcome`). Une question
regroupe ses issues via `namedOutcomes` + un `fallbackOutcome` catch-all. Les
seuils d'un marche a bandes se lisent dans `priceThresholds` de la question.

In [ ]:
def fields(desc):
    return dict(p.split(":", 1) for p in desc.split("|") if ":" in p)

# une question relie ses outcomes ; on y lit les seuils d'un marche a bandes
parent = {o: q for q in meta["questions"]
          for o in q["namedOutcomes"] + [q["fallbackOutcome"]]}

def label_for(o):
    f = fields(o["description"])
    try:
        if f.get("class") == "priceBinary":
            return f"{f['underlying']} > ${int(float(f['targetPrice'])):,}"
        if "index" in f and o["outcome"] in parent:
            qf = fields(parent[o["outcome"]]["description"])
            th = [int(float(t)) for t in qf["priceThresholds"].split(",")]
            i = int(f["index"])
            r = f"< ${th[0]:,}" if i == 0 else f"> ${th[-1]:,}" if i >= len(th) else f"${th[i-1]:,}-${th[i]:,}"
            return f"{qf['underlying']} {r}"
    except (KeyError, ValueError, IndexError):
        pass
    return o["name"]

for o in meta["outcomes"]:
    print(f'{o["outcome"]:>4}  {label_for(o)}')

## 4. Le numero d'outcome et les noms YES / NO

On prend un numero lu dans la liste, par exemple l'outcome 144 (le BTC up/down).
On en derive le nom du contrat (le *coin*) pour cibler un cote :

- **YES** = `#` suivi de `10 * outcome`
- **NO**  = `#` suivi de `10 * outcome + 1`

Pour l'outcome 144 : YES = `#1440`, NO = `#1441`.

In [ ]:
outcome_id = 144
yes_coin = f"#{10 * outcome_id}"      # YES -> #1440
no_coin  = f"#{10 * outcome_id + 1}"  # NO  -> #1441
print(yes_coin, no_coin)              # #1440 #1441

## 5. Envoyer un ordre limite

Quelques parametres :
- `PRICE` : prix par contrat entre 0.001 et 0.999, lu comme une probabilite
  (0.65 = payer 65 % de la valeur de reglement maximale).
- `TARGET_AMOUNT` : montant vise en USDH (notionnel minimum : 10).
- `SIZE` : Hyperliquid n'accepte que des contrats entiers, on arrondit au superieur.
- `tif: "Gtc"` (*good-till-cancelled*) : l'ordre reste dans le carnet jusqu'a
  son execution ou son annulation.

In [ ]:
COIN = "#1440"                           # YES de l'outcome 144 (BTC au-dessus de la cible)
PRICE = 0.65                             # prix par contrat (entre 0.001 et 0.999)
TARGET_AMOUNT = 12                       # montant vise en USDH (minimum : 10)
SIZE = math.ceil(TARGET_AMOUNT / PRICE)  # contrats entiers (ici 19)

result = hyperliquid.order(
    name=COIN,
    is_buy=True,                         # acheter le YES = prendre position sur l'issue
    sz=SIZE,
    limit_px=PRICE,
    order_type={"limit": {"tif": "Gtc"}},
)
result